# 01 — Data Exploration

**America's Development Paradox**: Exploring the World Bank MDG dataset (2006-2015) with a focus on U.S. data availability and peer comparisons.

In [ ]:
import sys
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from helpers import (
    load_data, load_metadata, get_country, get_countries,
    PEER_COUNTRIES, PEER_COUNTRIES_NO_US, AGGREGATES,
    FOCUS_VARIABLES, FOCUS_COLS, VAR_LABELS, COUNTRY_NAMES, COLORS
)

df = load_data()
print(f"Dataset shape: {df.shape}")
print(f"Years: {sorted(df['Year'].unique())}")
print(f"Unique entities: {df['Country.Code'].nunique()}")

## Data Availability for U.S. and Peers

Check which of our 14 focus variables have data for the U.S. and all 5 peer countries across 2006-2015.

In [ ]:
# Data availability: count non-NA values per country per variable
peers_df = get_countries(df, PEER_COUNTRIES)

availability = []
for col, label, goal, lib in FOCUS_VARIABLES:
    for code in PEER_COUNTRIES:
        country_data = peers_df[peers_df['Country.Code'] == code][col]
        n_valid = country_data.notna().sum()
        availability.append({
            'Variable': label,
            'Column': col,
            'Country': COUNTRY_NAMES.get(code, code),
            'Code': code,
            'Valid Years': n_valid,
            'Total Years': len(country_data),
        })

avail_df = pd.DataFrame(availability)
avail_pivot = avail_df.pivot(index='Variable', columns='Country', values='Valid Years')
avail_pivot = avail_pivot[['United States', 'United Kingdom', 'Germany', 'Japan', 'Canada', 'France']]
avail_pivot

In [ ]:
# Heatmap of data availability
fig = px.imshow(
    avail_pivot.values,
    x=avail_pivot.columns.tolist(),
    y=avail_pivot.index.tolist(),
    color_continuous_scale='Greens',
    labels={'color': 'Years with Data'},
    title='Data Availability: Focus Variables for U.S. and Peers (out of 10 years)',
    text_auto=True,
    aspect='auto',
)
fig.update_layout(height=600, width=900)
fig.show()

## U.S. Data Overview

Quick look at U.S. values for each focus variable across the decade.

In [ ]:
usa = get_country(df, 'USA')
usa_focus = usa[['Year'] + FOCUS_COLS].set_index('Year')
usa_focus.columns = [VAR_LABELS[c] for c in usa_focus.columns]
usa_focus.T

## 2010 Snapshot: U.S. vs. Peer Average

In [ ]:
year = 2010
usa_2010 = usa[usa['Year'] == year].iloc[0]

peers_2010 = df[
    (df['Country.Code'].isin(PEER_COUNTRIES_NO_US)) & 
    (df['Year'] == year)
]
peer_avg_2010 = peers_2010[FOCUS_COLS].mean()

snapshot = []
for col, label, goal, lower_is_better in FOCUS_VARIABLES:
    usa_val = usa_2010[col]
    peer_val = peer_avg_2010[col]
    if pd.notna(usa_val) and pd.notna(peer_val) and peer_val != 0:
        pct_diff = ((usa_val - peer_val) / abs(peer_val)) * 100
        # Determine if the difference is good or bad
        if lower_is_better:
            assessment = 'Worse' if usa_val > peer_val else 'Better'
        else:
            assessment = 'Better' if usa_val > peer_val else 'Worse'
        snapshot.append({
            'Indicator': label,
            'MDG': f'Goal {goal}',
            'USA': round(usa_val, 2),
            'Peer Avg': round(peer_val, 2),
            '% Diff': round(pct_diff, 1),
            'Assessment': assessment,
        })

snapshot_df = pd.DataFrame(snapshot)
snapshot_df.sort_values('% Diff', key=abs, ascending=False)

## U.S. vs. Income Group Averages

Compare U.S. values to High Income, Upper Middle Income, and World averages.

In [ ]:
agg_codes = ['HIC', 'UMC', 'WLD']
agg_2010 = df[(df['Country.Code'].isin(agg_codes)) & (df['Year'] == year)]

income_comparison = []
for col, label, goal, lower_is_better in FOCUS_VARIABLES:
    usa_val = usa_2010[col]
    if pd.isna(usa_val):
        continue
    
    row = {'Indicator': label, 'USA': round(usa_val, 2)}
    
    closest_group = None
    closest_dist = float('inf')
    
    for code in agg_codes:
        agg_row = agg_2010[agg_2010['Country.Code'] == code]
        if len(agg_row) > 0:
            agg_val = agg_row.iloc[0][col]
            if pd.notna(agg_val):
                row[AGGREGATES[code]] = round(agg_val, 2)
                dist = abs(usa_val - agg_val)
                if dist < closest_dist:
                    closest_dist = dist
                    closest_group = AGGREGATES[code]
    
    row['Closest To'] = closest_group
    income_comparison.append(row)

income_df = pd.DataFrame(income_comparison)
income_df

## U.S. Trends: 2006 vs. 2015

In [ ]:
usa_2006 = usa[usa['Year'] == 2006].iloc[0]
usa_2015 = usa[usa['Year'] == 2015].iloc[0]

trends = []
for col, label, goal, lower_is_better in FOCUS_VARIABLES:
    v06 = usa_2006[col]
    v15 = usa_2015[col]
    if pd.notna(v06) and pd.notna(v15) and v06 != 0:
        pct_change = ((v15 - v06) / abs(v06)) * 100
        if lower_is_better:
            direction = 'Improved' if v15 < v06 else 'Worsened'
        else:
            direction = 'Improved' if v15 > v06 else 'Worsened'
        trends.append({
            'Indicator': label,
            '2006': round(v06, 2),
            '2015': round(v15, 2),
            '% Change': round(pct_change, 1),
            'Direction': direction,
        })

trends_df = pd.DataFrame(trends)
trends_df.sort_values('% Change', key=abs, ascending=False)